
# Projeto: Funções de Escrita e Merge para Data Lake Delta

### Objetivo  
Este notebook centraliza funções utilitárias essenciais para gravação e atualização de dados no Data Lake utilizando formatos flexíveis (Delta, Parquet, CSV, TXT, XML) e para operações de upsert (merge) em tabelas Delta, garantindo consistência, padronização e controle temporal das cargas.

### Funções Principais:
- `salvar_no_datalake`: Função genérica para persistência de DataFrames em diversos formatos e modos, com opção de incluir coluna de data da carga.
- `inserir_com_merge`: Implementa lógica de upsert usando Delta Lake, unindo dados novos com dados existentes em tabelas Delta com base em chaves definidas.
- `adicionaColunasDeCarga`: Adiciona uma coluna `data_carga` com timestamp da carga ajustado para fuso horário de São Paulo.

## Observações

- A função `salvar_no_datalake` é o ponto central para gravação padronizada em diferentes formatos, respeitando modos de escrita e estrutura do Data Lake.
- `inserir_com_merge` utiliza a robustez do Delta Lake para garantir atomicidade e consistência nos processos de atualização incremental, evitando duplicidade e mantendo a integridade dos dados.
- O uso do `adicionaColunasDeCarga` é recomendado para monitoramento e auditoria de pipelines, facilitando rastreamento de quando os dados foram inseridos ou atualizados.

---

## Histórico de Atualizações

| Data       | Responsável    | Versão | Descrição                                                                 |
|------------|----------------|--------|---------------------------------------------------------------------------|
| 12/05/2025 | Lucas Zorn   | 1.0    | Criação das funções para gravação, merge e adição de data de carga.       |
| 19/05/2025 | Evelise Dias   | 1.1    | Inclusão de documentação                       |


In [0]:
def salvar_no_datalake(dataframe, path, modo="append", format="delta", data_carga: bool = False, saveAsTable: bool = False):
    """
    Grava um DataFrame no Data Lake no formato especificado, com opção de adicionar a coluna 'data_carga'.

    Args:
        dataframe (DataFrame): DataFrame a ser gravado.
        path (str): Caminho onde os dados serão gravados.
        modo (str): Modo de escrita ('append', 'overwrite', etc). Padrão: 'append'.
        format (str): Formato de escrita ('delta', 'parquet', 'csv', 'text'). Padrão: 'delta'.
        data_carga (bool): Se True, adiciona coluna de data de carga. Padrão: False.

    Returns:
        None
    """
    if data_carga:
        dataframe = adicionaColunasDeCarga(dataframe)

    if format == "csv":
        dataframe.write \
            .mode(modo) \
            .option("header", True) \
            .csv(path)

    elif format == "txt":
        dataframe.write \
            .format("text")\
            .mode(modo)\
            .save(path)

    elif format == "xml":
        dataframe.write\
            .format("text")\
            .option("rootTag", "registros")\
            .option("rowTag", "registro")\
            .mode(modo)\
            .save(path)

    elif format == "parquet":
        dataframe.write \
            .format("parquet")\
            .mode(modo)\
            .save(path)

    else:
        if saveAsTable:
            dataframe.write \
                .format(format) \
                .mode(modo) \
                .saveAsTable(path)
            return
        dataframe.write \
            .format(format) \
            .mode(modo) \
            .save(path)

In [0]:
def inserir_com_merge(dataframe, tabela_destino, camposChave=[], alias_origem="origem", alias_destino="destino"):
    """
    Realiza um upsert (merge) entre um DataFrame e uma tabela Delta com base nos campos-chave.

    Args:
        dataframe (DataFrame): DataFrame com os dados de origem.
        tabela_destino (str): Nome da tabela Delta de destino no banco (exemplo: "grupo01.bronze.nometabela").
        camposChave (list): Lista de campos usados como chave para identificar correspondência.
        alias_origem (str): Alias usado para o DataFrame de origem no merge.
        alias_destino (str): Alias usado para a tabela Delta no merge.

    Returns:
        None
    """

    if not spark.catalog.tableExists(tabela_destino):
        print("Tabela não existe. Criando como Delta.")
        dataframe.write \
            .format("delta") \
            .mode("overwrite") \
            .saveAsTable(tabela_destino)
        print("Tabela criada com sucesso.")
        return

    try:
        deltaTable = DeltaTable.forName(spark, tabela_destino)
    except Exception as e:
        raise Exception(f"A tabela '{tabela_destino}' existe mas não é uma Delta Table.\n")

    condicoes = " AND ".join([f"{alias_origem}.{campo} = {alias_destino}.{campo}" for campo in camposChave])

    deltaTable.alias(alias_destino).merge(
        source=dataframe.alias(alias_origem),
        condition=condicoes
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()

    print("Merge concluído com sucesso.")

In [0]:
from pyspark.sql.functions import current_timestamp, from_utc_timestamp, date_format

def adicionaColunasDeCarga(dataframe):
    dataframe_com_data_carga = dataframe.withColumn(
        "data_carga",
        date_format(from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"), "yyyy-MM-dd HH:mm:ss")
    )
    return dataframe_com_data_carga

In [0]:
from pyspark.sql.functions import sha2, concat_ws, coalesce, lit

def gera_primary_key(df):
    """
    Gera uma chave primária com hash SHA-256 baseado nas colunas dt_previsao, nom_municipio, 
    nom_estado e flg_fonte_previsao, adicionando-a como nova coluna 'id' ao DataFrame.
    
    Args:
        df: DataFrame do PySpark com as colunas dt_previsao, nom_municipio, nom_estado e flg_fonte_previsao.
    
    Returns:
        DataFrame com a nova coluna 'id'.
    """
    df_com_pk = df.withColumn(
        "id",
        sha2(
            concat_ws(
                "|",
                coalesce(df.dt_carga, lit("")),
                coalesce(df.dt_previsao, lit("")),
                coalesce(df.nom_municipio, lit("")),
                coalesce(df.sig_uf, lit("")),
                coalesce(df.flg_fonte_previsao, lit(""))
            ), 256
        )
    )
    return df_com_pk